# 🎵 Spotifake 2D CNN Spectrogram Genre Classifier — Direct Colab Auto-Downloader

This notebook trains a **2D Convolutional Neural Network (CNN)** on **Mel-Spectrogram images** generated from FMA + GTZAN audio tracks.

### ✨ No Google Drive Upload Needed!
This notebook automatically downloads both **GTZAN** and **FMA Medium (22 GB)** directly inside Google Colab's high-speed network (~100+ MB/s) in 3-5 minutes.

## ⚙️ Step 0 – GPU Check & Package Installation

In [ ]:
import subprocess, sys

# Install required packages quietly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torchaudio', 'torchvision', 'librosa', 'seaborn', 'pandas',
                'scikit-learn', 'matplotlib', 'tqdm'])

import torch
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

## ⚡ Step 1 – Automatic Direct Dataset Downloader (GTZAN + FMA Medium)

In [ ]:
import os
import subprocess
import glob
import pandas as pd

# Determine root directory depending on Colab or Kaggle environment
BASE_DIR = '/content' if os.path.exists('/content') else '/kaggle/working'

# ------------------------------------------------
# 1. Auto-Download GTZAN via HuggingFace (~1.2 GB, ~30s)
# ------------------------------------------------
gtzan_dir = os.path.join(BASE_DIR, 'gtzan_data')
os.makedirs(gtzan_dir, exist_ok=True)
if not any(glob.glob(f'{gtzan_dir}/**/*.wav', recursive=True)):
    print('⚡ Downloading GTZAN from HuggingFace...')
    tar_path = os.path.join(gtzan_dir, 'genres.tar.gz')
    subprocess.run(['wget', '-q', '-O', tar_path, 'https://huggingface.co/datasets/marsyas/gtzan/resolve/main/data/genres.tar.gz'])
    print('Extracting GTZAN...')
    subprocess.run(['tar', '-xzf', tar_path, '-C', gtzan_dir])
    if os.path.exists(tar_path):
        os.remove(tar_path)  # Cleanup archive to save space
    print('✅ GTZAN ready!')

# ------------------------------------------------
# 2. Auto-Download FMA Metadata & Generate Mapped Labels (~350 MB, ~5 sec)
# ------------------------------------------------
fma_dir = os.path.join(BASE_DIR, 'fma_data')
os.makedirs(fma_dir, exist_ok=True)
FMA_LABELS_PATH = os.path.join(fma_dir, 'mapped_labels.csv')

if not os.path.exists(FMA_LABELS_PATH):
    print('⚡ Downloading FMA metadata...')
    meta_zip = os.path.join(fma_dir, 'fma_metadata.zip')
    subprocess.run(['wget', '-q', '-O', meta_zip, 'https://os.unil.cloud.switch.ch/fma/fma_metadata.zip'])
    subprocess.run(['unzip', '-q', '-o', meta_zip, '-d', fma_dir])
    if os.path.exists(meta_zip):
        os.remove(meta_zip)
    
    print('Processing FMA metadata...')
    GENRE_MAP = {
        'Electronic': 'electronic', 'Experimental': 'ambient', 'Folk': 'folk',
        'Hip-Hop': 'hiphop', 'Instrumental': 'acoustic', 'International': 'indie',
        'Jazz': 'jazz', 'Classical': 'classical', 'Historic': 'ambient',
        'Country': 'country', 'Pop': 'pop', 'Rock': 'rock', 'Blues': 'blues',
        'Soul-RnB': 'soul', 'Easy Listening': 'ambient'
    }
    tracks_path = glob.glob(f'{fma_dir}/**/tracks.csv', recursive=True)[0]
    tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1])
    track_meta = tracks['track'].dropna(subset=['genre_top'])
    
    labels = []
    for track_id, row in track_meta.iterrows():
        fma_genre = row['genre_top']
        mapped = GENRE_MAP.get(fma_genre)
        if mapped:
            labels.append({'track_id': track_id, 'fma_genre': fma_genre, 'mapped_genre': mapped})
    
    pd.DataFrame(labels).to_csv(FMA_LABELS_PATH, index=False)
    print('✅ FMA metadata mapped!')

# ------------------------------------------------
# 3. Auto-Download FMA Medium Audio (~22 GB) & Auto-Delete Zip to Save Space
# ------------------------------------------------
USE_FMA = True
FMA_AUDIO_DIR = os.path.join(fma_dir, 'fma_medium')

if USE_FMA and not os.path.exists(FMA_AUDIO_DIR):
    print('⚡ Downloading FMA Medium Audio (~22 GB)... High-speed download in progress...')
    medium_zip = os.path.join(fma_dir, 'fma_medium.zip')
    subprocess.run(['wget', '-q', '-c', '-O', medium_zip, 'https://os.unil.cloud.switch.ch/fma/fma_medium.zip'])
    print('⚡ Unzipping FMA Medium Audio quietly...')
    subprocess.run(['unzip', '-q', '-o', medium_zip, '-d', fma_dir])
    if os.path.exists(medium_zip):
        print('🧹 Cleaning up fma_medium.zip to free 22.5 GB disk space...')
        os.remove(medium_zip)
    print('✅ FMA Medium Audio downloaded, unzipped, and disk space cleaned up!')

print('\n🚀 All Datasets Ready!')

## 🗂️ Step 2 – Build the File/Label Manifest

In [ ]:
import pandas as pd
from pathlib import Path
import glob

# ------------------------------------------------
# Build GTZAN file list (Dynamic search)
# ------------------------------------------------
gtzan_records = []
all_gtzan_wavs = glob.glob(f'{BASE_DIR}/gtzan_data/**/*.wav', recursive=True)
if not all_gtzan_wavs:
    all_gtzan_wavs = glob.glob(f'{BASE_DIR}/**/*.wav', recursive=True)

for wav_file in all_gtzan_wavs:
    genre = Path(wav_file).parent.name.lower().strip()
    if genre and not genre.startswith('.'):
        gtzan_records.append({
            'filepath': str(wav_file),
            'label': genre,
            'sample_weight': 10.0  # 10x boost for GTZAN quality
        })

df_gtzan = pd.DataFrame(gtzan_records)
print(f'GTZAN tracks loaded: {len(df_gtzan)}')
if len(df_gtzan) > 0:
    print(df_gtzan['label'].value_counts())

# ------------------------------------------------
# Build FMA file list
# ------------------------------------------------
df_fma = pd.DataFrame()

if USE_FMA:
    labels_df = pd.read_csv(FMA_LABELS_PATH)
    labels_df['mapped_genre'] = labels_df['mapped_genre'].replace({'hip-hop': 'hiphop'})
    valid_ids = set(labels_df['track_id'])

    fma_records = []
    for mp3 in glob.glob(str(Path(FMA_AUDIO_DIR) / '**/*.mp3'), recursive=True):
        try:
            tid = int(Path(mp3).stem)
            if tid in valid_ids:
                genre = labels_df.loc[labels_df['track_id'] == tid, 'mapped_genre'].values[0]
                fma_records.append({'filepath': mp3, 'label': genre, 'sample_weight': 1.0})
        except Exception:
            pass

    df_fma = pd.DataFrame(fma_records)
    if len(df_fma) > 0:
        # Cap FMA at 2000 tracks per genre
        df_fma = df_fma.groupby('label').head(2000).reset_index(drop=True)
        print(f'FMA tracks loaded: {len(df_fma)}')
        print(df_fma['label'].value_counts())

# ------------------------------------------------
# Merge both datasets
# ------------------------------------------------
dfs_to_concat = [df for df in [df_gtzan, df_fma] if len(df) > 0]
if not dfs_to_concat:
    raise RuntimeError(f'No tracks found! Check if audio files exist in {BASE_DIR}')

df_all = pd.concat(dfs_to_concat, ignore_index=True)
df_all['label'] = df_all['label'].str.lower().str.strip()

# Build label encoding
genres = sorted(df_all['label'].unique())
label2idx = {g: i for i, g in enumerate(genres)}
idx2label = {i: g for g, i in label2idx.items()}
df_all['label_idx'] = df_all['label'].map(label2idx)

print(f'\nTotal tracks: {len(df_all)} | Genres: {len(genres)}')
print('Genres:', genres)

## 🖼️ Step 3 – PyTorch Dataset: Audio → 2D Mel-Spectrogram

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Spectrogram parameters (must match exactly when running local inference!)
SAMPLE_RATE = 22050
CLIP_SECONDS = 30
N_MELS = 128         # Height of the 2D image
N_FFT = 2048
HOP_LENGTH = 512
TARGET_LENGTH = 1292  # Width of the 2D image (30s @ 22050 Hz / 512 hop)

mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS
)
db_transform = T.AmplitudeToDB(top_db=80)


class SpectrogramDataset(Dataset):
    def __init__(self, df, augment=False):
        self.records = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def _load_spectrogram(self, filepath):
        """Loads audio and converts to a 2D Mel-Spectrogram tensor (1, N_MELS, TIME)."""
        try:
            try:
                waveform, sr = torchaudio.load(filepath)
            except Exception:
                # Fallback to librosa for mp3 files
                import librosa
                y, sr = librosa.load(filepath, sr=SAMPLE_RATE, duration=CLIP_SECONDS, mono=True)
                waveform = torch.tensor(y).unsqueeze(0)

            # Resample if needed
            if sr != SAMPLE_RATE:
                waveform = T.Resample(sr, SAMPLE_RATE)(waveform)

            # Mix to mono
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)

            # Clip to CLIP_SECONDS from the middle of the song (50% offset)
            target_samples = SAMPLE_RATE * CLIP_SECONDS
            if waveform.shape[1] > target_samples:
                start = (waveform.shape[1] - target_samples) // 2
                waveform = waveform[:, start:start + target_samples]
            else:
                # Pad with zeros if song is shorter than 30s
                pad = target_samples - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, pad))

            # Convert to 2D Mel-Spectrogram
            mel = mel_transform(waveform)    # Shape: (1, N_MELS, TIME)
            mel = db_transform(mel)          # Convert to decibels

            # Normalize to [0, 1]
            mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)

            # Pad or crop width to fixed TARGET_LENGTH
            if mel.shape[2] < TARGET_LENGTH:
                mel = torch.nn.functional.pad(mel, (0, TARGET_LENGTH - mel.shape[2]))
            else:
                mel = mel[:, :, :TARGET_LENGTH]

            # Expand to 3 channels (ResNet expects RGB-like input)
            mel = mel.repeat(3, 1, 1)  # Shape: (3, N_MELS, TARGET_LENGTH)
            return mel

        except Exception:
            # Fallback for corrupt/unreadable audio files in FMA: return clean zero tensor so training never crashes
            return torch.zeros((3, N_MELS, TARGET_LENGTH))

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        mel = self._load_spectrogram(row['filepath'])
        label = int(row['label_idx'])
        weight = float(row.get('sample_weight', 1.0))
        return mel, label, weight


# Preview one spectrogram
import matplotlib.pyplot as plt

sample_ds = SpectrogramDataset(df_all.head(1))
mel_sample, lbl, wt = sample_ds[0]

plt.figure(figsize=(12, 3))
plt.imshow(mel_sample[0].numpy(), aspect='auto', origin='lower', cmap='magma')
plt.title(f'2D Mel-Spectrogram — Genre: {idx2label[lbl]}')
plt.xlabel('Time Frames')
plt.ylabel('Mel Frequency Bins')
plt.colorbar(label='dB (normalized)')
plt.tight_layout()
plt.show()
print(f'Spectrogram shape: {mel_sample.shape}  (Channels x Height x Width)')

## ✂️ Step 4 – Train / Validation Split & DataLoaders

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(
    df_all,
    test_size=0.2,
    stratify=df_all['label_idx'],
    random_state=42
)

print(f'Training samples: {len(df_train)} | Validation samples: {len(df_val)}')

train_ds = SpectrogramDataset(df_train, augment=True)
val_ds   = SpectrogramDataset(df_val,   augment=False)

BATCH_SIZE = 128

# Weighted sampler so the AI sees equal genre representation per batch
weights = df_train['sample_weight'].values.astype(float)
sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 🧠 Step 5 – Build the CNN Model (ResNet18 fine-tuned)

In [ ]:
import torch.nn as nn
import torchvision.models as models

NUM_CLASSES = len(genres)

# Load pretrained ResNet18 (trained on ImageNet)
cnn_model = models.resnet18(weights='DEFAULT')

# Replace the final fully-connected layer for our genre count
in_features = cnn_model.fc.in_features
cnn_model.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, NUM_CLASSES)
)

cnn_model = cnn_model.to(device)

total_params = sum(p.numel() for p in cnn_model.parameters())
trainable_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,}')
print(f'Output classes: {NUM_CLASSES} → {genres}')

## 🏋️ Step 6 – Training Loop

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

EPOCHS = 10
LR = 3e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(cnn_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_acc': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):
    # ------ Training ------
    cnn_model.train()
    train_loss = 0.0
    for mel, labels, weights in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False):
        mel, labels = mel.to(device), labels.to(device)
        weights = weights.float().to(device)

        optimizer.zero_grad()
        outputs = cnn_model(mel)

        # Apply per-sample weights to the loss
        loss = (criterion(outputs, labels) * weights).mean()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()

    # ------ Validation ------
    cnn_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for mel, labels, _ in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [Val]  ', leave=False):
            mel = mel.to(device)
            outputs = cnn_model(mel)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_f1  = f1_score(all_labels, all_preds, average='weighted')

    avg_loss = train_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {avg_loss:.4f} | '
          f'Val Acc: {val_acc:.4f} | '
          f'Val F1: {val_f1:.4f} | '
          f'LR: {scheduler.get_last_lr()[0]:.6f}')

print('\n✅ Training complete!')

## 📊 Step 7 – Training Curves & Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

# Training curves (if history exists)
if 'history' in locals() and len(history.get('train_loss', [])) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history['train_loss'], label='Train Loss', color='tomato')
    axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(history['val_acc'], label='Val Accuracy', color='steelblue')
    axes[1].plot(history['val_f1'],  label='Val F1 Score', color='seagreen')
    axes[1].set_title('Validation Metrics'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout()
    plt.show()

# Final Confusion Matrix
cnn_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for mel, labels, _ in val_loader:
        mel = mel.to(device)
        preds = cnn_model(mel).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=genres, yticklabels=genres)
plt.title('CNN Genre Classifier — Confusion Matrix')
plt.ylabel('True Genre'); plt.xlabel('Predicted Genre')
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=150)
plt.show()

final_acc = accuracy_score(all_labels, all_preds)
final_f1  = f1_score(all_labels, all_preds, average='weighted')
print(f'\n🎯 Final Validation Accuracy: {final_acc:.4f}')
print(f'🎯 Final Validation F1-Score:  {final_f1:.4f}')

## 💾 Step 8 – Save & Download the Model

In [ ]:
import json

# Save PyTorch model weights + metadata
save_bundle = {
    'model_state_dict': cnn_model.state_dict(),
    'genres': genres,
    'label2idx': label2idx,
    'idx2label': {str(k): v for k, v in idx2label.items()},
    'num_classes': NUM_CLASSES,
    'sample_rate': SAMPLE_RATE,
    'clip_seconds': CLIP_SECONDS,
    'n_mels': N_MELS,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'target_length': TARGET_LENGTH
}

torch.save(save_bundle, 'cnn_genre_model.pth')
print('✅ Successfully saved: cnn_genre_model.pth')

# Optional ONNX export wrapped in try/except
try:
    dummy_input = torch.randn(1, 3, N_MELS, TARGET_LENGTH).to(device)
    torch.onnx.export(
        cnn_model,
        dummy_input,
        'cnn_genre_model.onnx',
        input_names=['spectrogram'],
        output_names=['genre_logits'],
        dynamic_axes={'spectrogram': {0: 'batch_size'}}
    )
    print('Saved: cnn_genre_model.onnx')
except Exception as e:
    print(f'ONNX export skipped (PyTorch model .pth is saved and fully functional): {e}')

# Trigger download
try:
    from google.colab import files
    files.download('cnn_genre_model.pth')
    print('\n🎉 File downloaded! Place cnn_genre_model.pth in your BackendAI/ml_models/ folder.')
except Exception:
    print('\n🎉 File saved to working directory! You can download cnn_genre_model.pth from your sidebar.')